# Olist E-commerce: Complete Python Analysis
## Data Understanding • Data Cleaning • Feature Engineering • EDA

**Dataset:** Olist Brazilian E-Commerce Public Dataset (Kaggle)  
**Tools:** Python, pandas, NumPy and Matplotlib

### What will this notebook do?
We will use Python to understand the Olist data, check its quality, clean important fields, create delivery measures and explore business questions.

### Our important rule
We will keep the **nine cleaned CSV files separate** at the end. We will join tables temporarily only when an analysis needs information from more than one table. This keeps the files ready for Power BI data modelling.

### How to use this notebook
1. Run the code cells from top to bottom.
2. Read the short explanation before each code cell.
3. Check the displayed output before moving to the next section.
4. If an error appears, stop and fix that cell before continuing.

## 1. Understand the business problem

An online order passes through several steps: purchase, approval, seller preparation, carrier handover and delivery to the customer. Delays, freight charges and customer reviews can help an operations team understand where fulfilment performance needs attention.

**Main business question**

> What factors are associated with late delivery, freight cost variation and customer review scores in an e-commerce fulfilment operation?

We will investigate:
- How many delivered orders arrived later than the estimated date?
- How many days did delivery take?
- Do late orders have different review scores?
- Does late delivery vary by customer state?
- How do freight values and delivery performance vary across product categories?
- Where are sellers located?

**Important:** This is historical analysis. A pattern in the data does not automatically prove the reason for that pattern.

## 2. Import the Python libraries

A library is a collection of ready-made tools.

- `pandas` helps us work with tables.
- `numpy` helps with numerical operations.
- `matplotlib` helps us create charts.
- `Path` helps us locate files and folders.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os
import zipfile

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("Libraries imported successfully!")

## 3. Set the dataset folder

Kaggle attaches your dataset inside the notebook. The path below is the path used in your earlier file-inspection step.

**If Kaggle shows a different folder name:** open the notebook's Input panel, copy the dataset folder path and replace the path below.

In [ ]:
DATA_FOLDER = "/kaggle/input/datasets/olistbr/brazilian-ecommerce/"

# Check whether Kaggle can find the folder
print("Dataset folder found:", os.path.exists(DATA_FOLDER))

# Create a folder for our cleaned files
OUTPUT_FOLDER = "/kaggle/working/olist_cleaned_tables/"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("Cleaned files will be saved here:", OUTPUT_FOLDER)

## 4. Load the nine CSV files separately

A CSV file is a table. We will give every table a clear variable name.

**Why keep them separate?** Each file contains a different type of information. For example, the orders table describes orders, while the order-items table describes the products inside each order.

In [ ]:
customers = pd.read_csv(DATA_FOLDER + "olist_customers_dataset.csv")
geolocation = pd.read_csv(DATA_FOLDER + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(DATA_FOLDER + "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_FOLDER + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_FOLDER + "olist_order_reviews_dataset.csv")
orders = pd.read_csv(DATA_FOLDER + "olist_orders_dataset.csv")
products = pd.read_csv(DATA_FOLDER + "olist_products_dataset.csv")
sellers = pd.read_csv(DATA_FOLDER + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(DATA_FOLDER + "product_category_name_translation.csv")

print("All nine CSV files loaded.")

## 5. Understand the size of each table

`shape` gives us two numbers:
- First number = rows
- Second number = columns

We will print each table separately so it is easy to understand.

In [ ]:
print("Customers:", customers.shape)
print("Geolocation:", geolocation.shape)
print("Order items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)
print("Orders:", orders.shape)
print("Products:", products.shape)
print("Sellers:", sellers.shape)
print("Category translation:", category_translation.shape)

## 6. Look at the first five rows

`head()` shows the first five rows. It helps us understand what the columns and values look like.

We will start with the most important table for delivery analysis: `orders`.

In [ ]:
orders.head()

Now look at the customer and order-item tables. Notice that `customer_id` connects customers to orders, and `order_id` connects order items to orders.

In [ ]:
display(customers.head())
display(order_items.head())

We also need products, sellers, payments, reviews and category translations.

In [ ]:
display(products.head())
display(sellers.head())
display(payments.head())
display(reviews.head())
display(category_translation.head())

## 7. Understand the columns and data types

`info()` shows:
- Column names
- Number of non-empty values
- Data type of each column
- Approximate memory usage

Run it on the orders table first.

In [ ]:
orders.info()

Check the other tables too. You can run each line one by one.

In [ ]:
customers.info()
order_items.info()
payments.info()
reviews.info()
products.info()
sellers.info()
geolocation.info()
category_translation.info()

## 8. Check missing values

A missing value means that a cell has no recorded value. `isnull().sum()` counts missing cells column by column.

Do not delete every row that has a missing value. For example, an order without a delivery date may not have been delivered yet.

In [ ]:
print("Missing values in ORDERS")
display(orders.isnull().sum())

print("Missing values in REVIEWS")
display(reviews.isnull().sum())

print("Missing values in PRODUCTS")
display(products.isnull().sum())

Check whether the remaining tables contain missing values.

In [ ]:
print("Customers")
display(customers.isnull().sum())

print("Order items")
display(order_items.isnull().sum())

print("Payments")
display(payments.isnull().sum())

print("Sellers")
display(sellers.isnull().sum())

print("Geolocation")
display(geolocation.isnull().sum())

print("Category translation")
display(category_translation.isnull().sum())

## 9. Check exact duplicate rows

`duplicated().sum()` counts rows that are exact copies of an earlier row.

Zero exact duplicate rows does **not** automatically mean that every ID column is unique. We check important IDs separately in the next section.

In [ ]:
print("Customers duplicate rows:", customers.duplicated().sum())
print("Orders duplicate rows:", orders.duplicated().sum())
print("Order items duplicate rows:", order_items.duplicated().sum())
print("Payments duplicate rows:", payments.duplicated().sum())
print("Reviews duplicate rows:", reviews.duplicated().sum())
print("Products duplicate rows:", products.duplicated().sum())
print("Sellers duplicate rows:", sellers.duplicated().sum())
print("Geolocation duplicate rows:", geolocation.duplicated().sum())
print("Category translation duplicate rows:", category_translation.duplicated().sum())

## 10. Check important ID columns

An ID is used to identify a record. For example, `order_id` should identify one order in the orders table.

`duplicated(subset="column_name")` checks repeated values in a selected column.

Note: `customer_unique_id` can repeat because the same real customer may place orders more than once. We do not require it to be unique in the customer table.

In [ ]:
print("Repeated customer_id values:", customers["customer_id"].duplicated().sum())
print("Repeated order_id values:", orders["order_id"].duplicated().sum())
print("Repeated product_id values:", products["product_id"].duplicated().sum())
print("Repeated seller_id values:", sellers["seller_id"].duplicated().sum())

# An order can have multiple items. The combination of order_id and order_item_id identifies an order line.
print(
    "Repeated order-item combinations:",
    order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
)

# An order can have multiple payment records. Check the payment sequence with the order ID.
print(
    "Repeated payment combinations:",
    payments.duplicated(subset=["order_id", "payment_sequential"]).sum()
)

## 11. Understand the order status

Before cleaning missing dates, check the order statuses. Some orders may be cancelled or unavailable. Those orders may not have a customer delivery date.

We will keep all order statuses in the cleaned orders file. The delivery analysis will use delivered orders with valid dates.

In [ ]:
print("How many orders are in each status?")
display(orders["order_status"].value_counts())

### 11.1 Check missing delivery dates for delivered orders

This code looks only at orders marked as delivered. It helps us understand whether some delivered records still have missing timestamps.

In [ ]:
delivered_status_orders = orders[orders["order_status"] == "delivered"]

print("Delivered orders:", len(delivered_status_orders))
display(delivered_status_orders.isnull().sum())

### 11.2 Check missing dates for orders that are not delivered

These missing dates may be expected because the order did not reach the customer.

In [ ]:
not_delivered_orders = orders[orders["order_status"] != "delivered"]

print("Orders with a status other than delivered:", len(not_delivered_orders))
display(not_delivered_orders.isnull().sum())

## 12. Make a clean working copy of the tables

We will not change the original DataFrames directly. Instead, we create separate working copies. This is a simple way to keep the source version available for comparison.

In [ ]:
customers_clean = customers.copy()
geolocation_clean = geolocation.copy()
order_items_clean = order_items.copy()
payments_clean = payments.copy()
reviews_clean = reviews.copy()
orders_clean = orders.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
category_translation_clean = category_translation.copy()

print("Working copies created.")

## 13. Convert date columns into date format

Date columns were loaded as text (`object`). Python needs date format to subtract dates and calculate delivery time.

`pd.to_datetime()` converts text into dates. `errors="coerce"` means an invalid date becomes a missing date (`NaT`) instead of stopping the notebook.

In [ ]:
orders_clean["order_purchase_timestamp"] = pd.to_datetime(
    orders_clean["order_purchase_timestamp"], errors="coerce"
)

orders_clean["order_approved_at"] = pd.to_datetime(
    orders_clean["order_approved_at"], errors="coerce"
)

orders_clean["order_delivered_carrier_date"] = pd.to_datetime(
    orders_clean["order_delivered_carrier_date"], errors="coerce"
)

orders_clean["order_delivered_customer_date"] = pd.to_datetime(
    orders_clean["order_delivered_customer_date"], errors="coerce"
)

orders_clean["order_estimated_delivery_date"] = pd.to_datetime(
    orders_clean["order_estimated_delivery_date"], errors="coerce"
)

print("Order date columns converted.")

Convert the shipping and review date columns as well.

In [ ]:
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"], errors="coerce"
)

reviews_clean["review_creation_date"] = pd.to_datetime(
    reviews_clean["review_creation_date"], errors="coerce"
)

reviews_clean["review_answer_timestamp"] = pd.to_datetime(
    reviews_clean["review_answer_timestamp"], errors="coerce"
)

print("Shipping and review dates converted.")

## 14. Clean text columns

Sometimes text values contain extra spaces at the beginning or end. `str.strip()` removes those spaces.

We are not changing IDs or replacing missing values with made-up information.

In [ ]:
customers_clean["customer_city"] = customers_clean["customer_city"].str.strip()
customers_clean["customer_state"] = customers_clean["customer_state"].str.strip()

sellers_clean["seller_city"] = sellers_clean["seller_city"].str.strip()
sellers_clean["seller_state"] = sellers_clean["seller_state"].str.strip()

products_clean["product_category_name"] = products_clean["product_category_name"].str.strip()

category_translation_clean["product_category_name"] = (
    category_translation_clean["product_category_name"].str.strip()
)
category_translation_clean["product_category_name_english"] = (
    category_translation_clean["product_category_name_english"].str.strip()
)

print("Selected text columns cleaned.")

## 15. Check the cleaned dates and missing values again

We are checking the results of date conversion. Missing values are not automatically removed; we need to understand why they are missing.

In [ ]:
print("Missing order dates after conversion:")
display(orders_clean[[
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]].isnull().sum())

print("Missing review dates after conversion:")
display(reviews_clean[[
    "review_creation_date",
    "review_answer_timestamp"
]].isnull().sum())

## 16. Create delivery performance columns

We will add three useful columns to the orders table.

1. **delivery_days:** actual customer delivery date minus purchase date.
2. **delay_days:** actual customer delivery date minus estimated delivery date.
3. **late_delivery_flag:** 1 means late, 0 means on time or early. Missing means we cannot calculate delivery status.

A positive `delay_days` value means delivery happened after the estimated date.

In [ ]:
# Total number of days from purchase to customer delivery
orders_clean["delivery_days"] = (
    orders_clean["order_delivered_customer_date"] -
    orders_clean["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Number of days early or late compared with the estimated date
orders_clean["delay_days"] = (
    orders_clean["order_delivered_customer_date"] -
    orders_clean["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

# Start by comparing actual delay with zero days
orders_clean["late_delivery_flag"] = (
    orders_clean["delay_days"] > 0
).astype("Int64")

# If delay_days is missing, we cannot say whether the order was late
orders_clean.loc[
    orders_clean["delay_days"].isnull(),
    "late_delivery_flag"
] = pd.NA

print("Delivery columns created.")
display(orders_clean[[
    "order_id",
    "order_status",
    "delivery_days",
    "delay_days",
    "late_delivery_flag"
]].head(10))

### 16.1 Select delivered orders that have the dates we need

For late-delivery analysis, we need:
- Order status = delivered
- Actual customer delivery date
- Estimated delivery date
- Purchase date

Orders without these dates remain in the cleaned file but are not included in the late-delivery calculation.

In [ ]:
delivered_analysis = orders_clean[
    (orders_clean["order_status"] == "delivered") &
    (orders_clean["order_delivered_customer_date"].notnull()) &
    (orders_clean["order_estimated_delivery_date"].notnull()) &
    (orders_clean["order_purchase_timestamp"].notnull())
].copy()

print("All orders:", len(orders_clean))
print("Delivered orders with required dates:", len(delivered_analysis))

## 17. Create small summary tables before joining

Some tables contain multiple rows for one order:
- An order can contain several items.
- An order can have multiple payment records.
- An order can have review records.

If we join all these raw tables directly, one order can turn into many repeated rows. That can inflate totals.

To avoid this, we first create one summary row per order for payments, reviews and order items.

### 17.1 Payment summary — one row per order

`groupby("order_id")` puts payment rows into order groups. `sum()` adds payment values within each order.

In [ ]:
payments_by_order = payments_clean.groupby("order_id")["payment_value"].sum().reset_index()

payments_by_order = payments_by_order.rename(
    columns={"payment_value": "total_payment_value"}
)

display(payments_by_order.head())

### 17.2 Review summary — one row per order

We calculate the average review score for each order. We also count the review records.

In [ ]:
reviews_by_order = reviews_clean.groupby("order_id").agg({
    "review_score": "mean",
    "review_id": "count"
}).reset_index()

reviews_by_order = reviews_by_order.rename(columns={
    "review_score": "average_review_score",
    "review_id": "review_count"
})

display(reviews_by_order.head())

### 17.3 Order-item summary — one row per order

An order can contain more than one product item. We calculate item count, number of different products, total item price and total freight.

In [ ]:
items_by_order = order_items_clean.groupby("order_id").agg({
    "order_item_id": "count",
    "product_id": "nunique",
    "price": "sum",
    "freight_value": "sum",
    "seller_id": "nunique"
}).reset_index()

items_by_order = items_by_order.rename(columns={
    "order_item_id": "item_count",
    "product_id": "different_products",
    "price": "total_item_price",
    "freight_value": "total_freight_value",
    "seller_id": "seller_count"
})

display(items_by_order.head())

## 18. Join tables for order-level analysis

A join combines information from related tables.

We will start with orders and add:
- Customer state/city
- Item and freight summary
- Payment summary
- Review summary

These joins create a temporary analysis table. We will **not** export this as a replacement for the nine separate Power BI tables.

In [ ]:
# Add customer location to each order
analysis_orders = orders_clean.merge(
    customers_clean[["customer_id", "customer_city", "customer_state"]],
    on="customer_id",
    how="left"
)

# Add the item summary
analysis_orders = analysis_orders.merge(
    items_by_order,
    on="order_id",
    how="left"
)

# Add the payment summary
analysis_orders = analysis_orders.merge(
    payments_by_order,
    on="order_id",
    how="left"
)

# Add the review summary
analysis_orders = analysis_orders.merge(
    reviews_by_order,
    on="order_id",
    how="left"
)

print("Orders table rows:", len(orders_clean))
print("Joined analysis table rows:", len(analysis_orders))
display(analysis_orders.head())

**Check:** The joined table should have the same number of rows as the orders table because we added summaries that contain one row per order. If the row count increases, stop and investigate the join keys.

In [ ]:
print("Do row counts match?", len(orders_clean) == len(analysis_orders))

## 19. Calculate the main delivery KPIs

We use only delivered orders with valid delivery and estimated dates.

The late-delivery rate is:

**Late delivery rate = Late delivered orders ÷ eligible delivered orders × 100**

This denominator is important. We do not include orders that were never delivered or do not have the required dates.

In [ ]:
eligible_orders = analysis_orders[
    (analysis_orders["order_status"] == "delivered") &
    (analysis_orders["order_delivered_customer_date"].notnull()) &
    (analysis_orders["order_estimated_delivery_date"].notnull()) &
    (analysis_orders["late_delivery_flag"].notnull())
].copy()

total_eligible_orders = eligible_orders["order_id"].nunique()

late_orders = eligible_orders[
    eligible_orders["late_delivery_flag"] == 1
]["order_id"].nunique()

on_time_orders = eligible_orders[
    eligible_orders["late_delivery_flag"] == 0
]["order_id"].nunique()

late_delivery_rate = late_orders / total_eligible_orders * 100
on_time_delivery_rate = on_time_orders / total_eligible_orders * 100

average_delivery_days = eligible_orders["delivery_days"].mean()
median_delivery_days = eligible_orders["delivery_days"].median()
average_review_score = eligible_orders["average_review_score"].mean()
average_freight_per_order = eligible_orders["total_freight_value"].mean()

print("Eligible delivered orders:", total_eligible_orders)
print("Late orders:", late_orders)
print("On-time / early orders:", on_time_orders)
print("Late delivery rate (%):", round(late_delivery_rate, 2))
print("On-time delivery rate (%):", round(on_time_delivery_rate, 2))
print("Average delivery days:", round(average_delivery_days, 2))
print("Median delivery days:", round(median_delivery_days, 2))
print("Average review score:", round(average_review_score, 2))
print("Average freight per order:", round(average_freight_per_order, 2))

## 20. EDA — Delivery time distribution

EDA means Exploratory Data Analysis. We use summaries and charts to understand patterns in the data.

This histogram shows how many delivered orders fall into different delivery-time ranges.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(eligible_orders["delivery_days"].dropna(), bins=30, edgecolor="black")
plt.title("Distribution of Delivery Days")
plt.xlabel("Days from purchase to customer delivery")
plt.ylabel("Number of orders")
plt.show()

## 21. EDA — Late vs on-time orders

This chart compares the number of late orders with the number of orders delivered on or before the estimated date.

In [ ]:
delivery_counts = eligible_orders["late_delivery_flag"].value_counts().sort_index()

plt.figure(figsize=(7, 5))
plt.bar(
    ["On time / early", "Late"],
    [delivery_counts.get(0, 0), delivery_counts.get(1, 0)]
)
plt.title("On-time and Late Delivered Orders")
plt.ylabel("Number of orders")
plt.show()

## 22. EDA — Review score and delivery status

We compare the average review score for late orders and orders delivered on time or early.

This comparison shows an association. It does not prove that delivery timing alone caused a review score.

In [ ]:
review_data = eligible_orders.dropna(subset=["average_review_score"]).copy()

review_summary = review_data.groupby("late_delivery_flag")["average_review_score"].agg(
    ["count", "mean", "median"]
).reset_index()

review_summary["delivery_status"] = review_summary["late_delivery_flag"].map({
    0: "On time / early",
    1: "Late"
})

display(review_summary)

plt.figure(figsize=(7, 5))
plt.bar(
    review_summary["delivery_status"],
    review_summary["mean"]
)
plt.title("Average Review Score by Delivery Status")
plt.xlabel("")
plt.ylabel("Average review score")
plt.ylim(0, 5)
plt.show()

## 23. EDA — Delivery performance by customer state

We compare states using:
- Number of eligible delivered orders
- Number of late orders
- Late-delivery rate
- Average delivery days

Always look at the order count with the percentage. A rate based on a small number of orders can change a lot.

In [ ]:
state_summary = eligible_orders.groupby("customer_state").agg({
    "order_id": "nunique",
    "late_delivery_flag": "mean",
    "delivery_days": "mean"
}).reset_index()

state_summary = state_summary.rename(columns={
    "order_id": "eligible_orders",
    "late_delivery_flag": "late_delivery_rate",
    "delivery_days": "average_delivery_days"
})

state_summary["late_delivery_rate"] = state_summary["late_delivery_rate"] * 100
state_summary = state_summary.sort_values("eligible_orders", ascending=False)

display(state_summary.head(15))

The next chart shows the late-delivery rate for the 15 states with the largest number of eligible delivered orders.

In [ ]:
top_15_states = state_summary.head(15).sort_values("late_delivery_rate")

plt.figure(figsize=(10, 6))
plt.barh(
    top_15_states["customer_state"],
    top_15_states["late_delivery_rate"]
)
plt.title("Late Delivery Rate — 15 Highest-Volume Customer States")
plt.xlabel("Late delivery rate (%)")
plt.ylabel("Customer state")
plt.show()

## 24. EDA — Product category and freight

This analysis starts with order items, then adds delivery information and product categories.

**Important:** One order can contain several items. The category table therefore measures item-level/category-level exposure. Its late rate is the percentage of category items that belong to late delivered orders, not a unique-order rate.

In [ ]:
# Add order delivery information to each order item
item_analysis = order_items_clean.merge(
    orders_clean[[
        "order_id",
        "order_status",
        "late_delivery_flag",
        "delivery_days"
    ]],
    on="order_id",
    how="left"
)

# Add product category
item_analysis = item_analysis.merge(
    products_clean[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

# Add English category names
item_analysis = item_analysis.merge(
    category_translation_clean,
    on="product_category_name",
    how="left"
)

# Keep delivered items that have a calculable late-delivery flag
delivered_items = item_analysis[
    (item_analysis["order_status"] == "delivered") &
    (item_analysis["late_delivery_flag"].notnull())
].copy()

# Give missing category translations a clear label for this analysis only
delivered_items["product_category_name_english"] = (
    delivered_items["product_category_name_english"]
    .fillna("Unknown / not translated")
)

display(delivered_items.head())

Now group the items by English product category. We calculate item count, number of orders, average freight per item and the late-item rate.

In [ ]:
category_summary = delivered_items.groupby("product_category_name_english").agg({
    "order_item_id": "count",
    "order_id": "nunique",
    "freight_value": "mean",
    "late_delivery_flag": "mean"
}).reset_index()

category_summary = category_summary.rename(columns={
    "order_item_id": "number_of_items",
    "order_id": "number_of_orders",
    "freight_value": "average_freight_per_item",
    "late_delivery_flag": "late_item_rate"
})

category_summary["late_item_rate"] = category_summary["late_item_rate"] * 100
category_summary = category_summary.sort_values("number_of_orders", ascending=False)

display(category_summary.head(20))

The next chart compares average freight per item for the 15 categories with the most delivered orders.

In [ ]:
top_categories = category_summary.head(15).sort_values("average_freight_per_item")

plt.figure(figsize=(10, 6))
plt.barh(
    top_categories["product_category_name_english"],
    top_categories["average_freight_per_item"]
)
plt.title("Average Freight per Item — 15 Highest-Volume Categories")
plt.xlabel("Average freight value per item (dataset currency units)")
plt.ylabel("Product category")
plt.show()

## 25. EDA — Seller state concentration

Seller location is connected to order items through `seller_id`. We count order items, distinct orders and distinct sellers for each seller state.

In [ ]:
seller_item_data = order_items_clean.merge(
    sellers_clean[["seller_id", "seller_state"]],
    on="seller_id",
    how="left"
)

seller_state_summary = seller_item_data.groupby("seller_state").agg({
    "order_item_id": "count",
    "order_id": "nunique",
    "seller_id": "nunique",
    "freight_value": "sum"
}).reset_index()

seller_state_summary = seller_state_summary.rename(columns={
    "order_item_id": "number_of_items",
    "order_id": "number_of_orders",
    "seller_id": "number_of_sellers",
    "freight_value": "total_freight_value"
})

seller_state_summary = seller_state_summary.sort_values(
    "number_of_items", ascending=False
)

display(seller_state_summary)

## 26. EDA — Freight value distribution

This histogram shows how freight values are distributed across individual order items. The second chart compares order-level freight total with delivery time.

The freight amount is shown in the dataset's recorded currency units. Confirm the currency convention before adding a currency symbol in a report.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(order_items_clean["freight_value"].dropna(), bins=40, edgecolor="black")
plt.title("Freight Value per Order Item")
plt.xlabel("Freight value (dataset currency units)")
plt.ylabel("Number of order items")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    eligible_orders["total_freight_value"],
    eligible_orders["delivery_days"],
    alpha=0.2
)
plt.title("Order Freight Total vs Delivery Days")
plt.xlabel("Total freight value per order")
plt.ylabel("Delivery days")
plt.show()

## 27. Prepare the nine separate cleaned CSV files

This is an important project requirement.

We will export:
1. Customers
2. Geolocation
3. Order items
4. Payments
5. Reviews
6. Orders
7. Products
8. Sellers
9. Category translation

The temporary analysis tables created by merging are **not** used as the Power BI source model.

### 27.1 Prepare the geolocation table

The raw geolocation table can contain several coordinate records for one ZIP prefix. For a simple Power BI lookup, we will create one row per ZIP prefix:
- Latitude = average latitude for that ZIP prefix
- Longitude = average longitude for that ZIP prefix
- City and state = first available value

This is an approximate ZIP-prefix lookup, not an exact address location.

In [ ]:
geolocation_clean = geolocation_clean.dropna(
    subset=["geolocation_zip_code_prefix"]
)

geolocation_clean = geolocation_clean.groupby(
    "geolocation_zip_code_prefix"
).agg({
    "geolocation_lat": "mean",
    "geolocation_lng": "mean",
    "geolocation_city": "first",
    "geolocation_state": "first"
}).reset_index()

print("Geolocation rows after grouping:", len(geolocation_clean))
display(geolocation_clean.head())

### 27.2 Save each cleaned table as its own CSV file

`to_csv()` saves a DataFrame as a CSV file. `index=False` prevents pandas from adding an extra row-number column.

In [ ]:
customers_clean.to_csv(OUTPUT_FOLDER + "customers_clean.csv", index=False)
geolocation_clean.to_csv(OUTPUT_FOLDER + "geolocation_clean.csv", index=False)
order_items_clean.to_csv(OUTPUT_FOLDER + "order_items_clean.csv", index=False)
payments_clean.to_csv(OUTPUT_FOLDER + "payments_clean.csv", index=False)
reviews_clean.to_csv(OUTPUT_FOLDER + "reviews_clean.csv", index=False)
orders_clean.to_csv(OUTPUT_FOLDER + "orders_clean.csv", index=False)
products_clean.to_csv(OUTPUT_FOLDER + "products_clean.csv", index=False)
sellers_clean.to_csv(OUTPUT_FOLDER + "sellers_clean.csv", index=False)
category_translation_clean.to_csv(OUTPUT_FOLDER + "category_translation_clean.csv", index=False)

print("All nine cleaned CSV files have been saved.")

### 27.3 Save the EDA summary tables

These extra files are optional reporting summaries. They do not replace the nine relational tables.

In [ ]:
state_summary.to_csv(OUTPUT_FOLDER + "eda_state_delivery_summary.csv", index=False)
category_summary.to_csv(OUTPUT_FOLDER + "eda_category_summary.csv", index=False)
seller_state_summary.to_csv(OUTPUT_FOLDER + "eda_seller_state_summary.csv", index=False)
review_summary.to_csv(OUTPUT_FOLDER + "eda_review_delivery_summary.csv", index=False)

print("EDA summary CSV files saved.")

### 27.4 Check that the nine files exist

We will list the files in the output folder and check their file sizes.

In [ ]:
output_files = os.listdir(OUTPUT_FOLDER)

for file_name in output_files:
    file_path = OUTPUT_FOLDER + file_name
    file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(file_name, "-", round(file_size_mb, 2), "MB")

### 27.5 Create one ZIP package for download

The ZIP file contains the nine cleaned CSV files and the optional EDA summaries. Download it from the Kaggle notebook's Output panel.

In [ ]:
zip_file_path = "/kaggle/working/Olist_Cleaned_Tables_and_EDA.zip"

with zipfile.ZipFile(zip_file_path, "w") as zip_file:
    for file_name in output_files:
        full_file_path = OUTPUT_FOLDER + file_name
        zip_file.write(full_file_path, arcname=file_name)

print("ZIP package created:", zip_file_path)

## 28. Power BI modelling guidance

Keep the nine tables separate. Suggested relationships:

| From table | Column | To table | Column |
|---|---|---|---|
| customers | customer_id | orders | customer_id |
| orders | order_id | order_items | order_id |
| orders | order_id | payments | order_id |
| orders | order_id | reviews | order_id |
| products | product_id | order_items | product_id |
| sellers | seller_id | order_items | seller_id |
| category_translation | product_category_name | products | product_category_name |

Geolocation can be used as a ZIP-prefix lookup for customer and seller locations. Because both customer and seller tables use ZIP prefixes, avoid creating ambiguous relationships. You may need separate customer-geolocation and seller-geolocation lookup tables in Power BI.

**Do not merge all nine tables into one giant table in Power BI.** Orders, order items, payments and reviews have different levels of detail. Keeping their natural grains helps avoid double-counting.

## 29. Final project checklist

After running the notebook, check that:

- [ ] All nine source tables loaded successfully.
- [ ] Missing values and duplicate rows were inspected.
- [ ] Order status was checked before interpreting missing delivery dates.
- [ ] Date columns were converted.
- [ ] Delivery days, delay days and late-delivery flag were created.
- [ ] Payments, reviews and order items were summarised before order-level joins.
- [ ] EDA tables and charts were generated.
- [ ] Nine cleaned CSV files were saved separately.
- [ ] The ZIP package was created successfully.

### Limitations to remember
- This is historical data, not live delivery tracking.
- Late delivery is calculated only when actual and estimated delivery dates are available for a delivered order.
- Review differences show association, not proof of causation.
- Freight values should be labelled in dataset currency units until the currency convention is verified.
- State/category comparisons should show the supporting order or item counts.